<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module2_Labs/Lab5_Cost_Hamiltonian_MaxCut.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 5 — Encoding Max-Cut: The Cost Hamiltonian
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Formulate the **Max-Cut problem** as a Hamiltonian $H_C$
2. Build the complete **cost operator** $U_C(\gamma) = e^{-i\gamma H_C}$ for all 6 edges
3. Verify that $U_C$ assigns phases **proportional to the cut value** of each bitstring
4. Understand why $\gamma$ must be tuned (too small → no signal; too large → wraps)
5. Visualize the **phase landscape** over all 32 candidate solutions

---
### 📖 Background: Cost Hamiltonian

For Max-Cut, the cost Hamiltonian for edge $(i,j)$ is:
$$H_{C,ij} = \frac{1}{2}(I - Z_iZ_j)$$

This gives **eigenvalue 0** when nodes $i,j$ are in the **same** group (edge not cut), and **eigenvalue 1** when they are in **different** groups (edge cut).

The total cost Hamiltonian sums over all edges:
$$H_C = \sum_{(i,j)\in E} \frac{1}{2}(I - Z_iZ_j)$$

The **cost operator** is the unitary:
$$U_C(\gamma) = e^{-i\gamma H_C} = \prod_{(i,j)\in E} e^{-i\gamma\frac{1}{2}(I-Z_iZ_j)} \propto \prod_{(i,j)\in E} R_{ZZ}(-\gamma)$$

For the 5-node graph in our course, edges are: $(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)$

---

In [ ]:
#Library installations if needed
!pip install qiskit qiskit-aer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()

# ── Graph definition ─────────────────────────────────────────────────────────
N_NODES = 5
EDGES   = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

# ── Helper: compute cut value for a bitstring ─────────────────────────────────
def cut_value(bitstring, edges):
    return sum(1 for u,v in edges if bitstring[u] != bitstring[v])

# ── All 32 candidate bitstrings ───────────────────────────────────────────────
all_bitstrings = [format(i, '05b') for i in range(32)]
all_cuts = [cut_value(bs, EDGES) for bs in all_bitstrings]

print("5-node Max-Cut Problem")
print(f"Edges: {EDGES}")
print(f"Total bitstrings: {len(all_bitstrings)}")
print(f"Max cut value: {max(all_cuts)}")
print(f"Optimal bitstrings: {[bs for bs,c in zip(all_bitstrings,all_cuts) if c==max(all_cuts)]}")

In [ ]:
# ── Visualize the graph ───────────────────────────────────────────────────────
G = nx.Graph()
G.add_nodes_from(range(N_NODES))
G.add_edges_from(EDGES)

pos = {0:(0,1), 1:(0,0), 2:(0,-1), 3:(2,0.5), 4:(2,-0.5)}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# All same group
for ax, partition, title in [
    (axes[0], '00000', 'All same group: cut=0'),
    (axes[1], '01001', 'Bitstring 01001: cut=3'),
    (axes[2], '00011', 'Optimal 00011: cut=6'),
]:
    colors_nodes = ['steelblue' if partition[i]=='0' else 'tomato' for i in range(5)]
    edge_colors  = ['red' if partition[u]!=partition[v] else 'gray' for u,v in EDGES]
    edge_widths  = [3 if partition[u]!=partition[v] else 1 for u,v in EDGES]

    nx.draw(G, pos=pos, ax=ax, node_color=colors_nodes,
            edge_color=edge_colors, width=edge_widths,
            with_labels=True, node_size=600, font_color='white', font_weight='bold')
    ax.set_title(title, fontsize=11)
    ax.text(1, -1.3, f'cut={cut_value(partition,EDGES)}',
            ha='center', fontsize=12, color='darkred', fontweight='bold')

plt.suptitle('5-Node Max-Cut Graph\n(Blue=group 0, Red=group 1, Red edges=cut)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 2: Building the Cost Operator

In [ ]:
# ── 2.1  Build the Cost Operator Circuit ─────────────────────────────────────
def cost_operator_circuit(gamma, n_qubits, edges):
    """
    Build U_C(γ) = prod_{(i,j)∈E} R_ZZ(-γ) over all edges.
    Uses CNOT–R_Z–CNOT decomposition for each edge.
    """
    qc = QuantumCircuit(n_qubits)
    for u, v in edges:
        qc.cx(u, v)           # CNOT
        qc.rz(-gamma, v)      # R_Z(-γ) — negative sign from Hamiltonian formulation
        qc.cx(u, v)           # CNOT
    return qc

gamma = np.pi/6   # 30° per unit of cut
qc_uc = cost_operator_circuit(gamma, N_NODES, EDGES)

print("Cost Operator U_C(γ=π/6) Circuit:")
print(qc_uc.draw('text'))

In [ ]:
# ── 2.2  Verify: apply H then U_C, check that phase ∝ cut value ──────────────
gamma = np.pi/6

# Full circuit: H^n → U_C
qc_verify = QuantumCircuit(N_NODES)
qc_verify.h(range(N_NODES))            # Step 1: Everyone on stage
qc_verify.compose(cost_operator_circuit(gamma, N_NODES, EDGES), inplace=True)

sv = Statevector(qc_verify)

print(f"Phase encoding verification (γ = π/6 = 30°/unit):")
print(f"\n{'Bitstring':>10} | {'Phase (deg)':>12} | {'Cut value':>10} | {'Phase/γ':>10} | {'Match?':>7}")
print("-" * 60)

gamma_deg = np.degrees(gamma)
for bs, amp in zip(all_bitstrings, sv.data):
    phase = np.degrees(np.angle(amp))
    cv    = cut_value(bs, EDGES)
    # Expected phase: cut states get positive, non-cut get negative
    # Combined: phase = γ*(2*cut - n_edges) / 2  (from Hamiltonian)
    n_edges = len(EDGES)
    expected_phase = np.degrees(gamma/2) * (2*cv - n_edges)
    match = abs(phase - expected_phase) < 0.5
    if cv in [0, 3, 6]:  # show representative cut values
        print(f"{bs:>10} | {phase:>10.2f}° | {cv:>10} | {phase/gamma_deg:>10.2f} | {'✓' if match else '✗'}")

In [ ]:
# ── 2.3  Visualize the phase landscape ───────────────────────────────────────
phases    = np.degrees([np.angle(amp) for amp in sv.data])
probs     = [abs(amp)**2 for amp in sv.data]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Phases
bar_colors = plt.cm.RdYlGn([c/6 for c in all_cuts])
axes[0].bar(range(32), phases, color=bar_colors)
axes[0].set_xlabel('Bitstring index (0-31)')
axes[0].set_ylabel('Phase (degrees)')
axes[0].set_title('Phase assigned by U_C(γ=π/6) — proportional to cut value')
axes[0].axhline(0, color='black', lw=0.5)
# Add legend
for cv, label in [(0,'cut=0 (no cut)'), (3,'cut=3 (mid)'), (6,'cut=6 (optimal)')]:
    color = plt.cm.RdYlGn(cv/6)
    axes[0].bar(0, 0, color=color, label=label)  # dummy bar for legend
axes[0].legend(loc='upper right')

# Plot 2: Probabilities (should be flat — phase doesn't change probs)
axes[1].bar(range(32), probs, color='steelblue')
axes[1].axhline(1/32, color='red', linestyle='--', label=f'Expected = {1/32:.4f}')
axes[1].set_xlabel('Bitstring index (0-31)')
axes[1].set_ylabel('Probability')
axes[1].set_title('Probabilities after H → U_C — still uniform! (Phase ≠ Probability)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 3: Effect of γ — Too Small, Just Right, Too Large

In [ ]:
# ── 3.1  Show how γ affects the phase spread ──────────────────────────────────
gamma_values = [0.1, np.pi/6, np.pi/3, np.pi, 2*np.pi]
gamma_labels = ['γ=0.1 (too small)', 'γ=π/6', 'γ=π/3', 'γ=π', 'γ=2π (wraps)']

fig, axes = plt.subplots(1, len(gamma_values), figsize=(18, 4))

for ax, gamma_val, gamma_lbl in zip(axes, gamma_values, gamma_labels):
    qc = QuantumCircuit(N_NODES)
    qc.h(range(N_NODES))
    qc.compose(cost_operator_circuit(gamma_val, N_NODES, EDGES), inplace=True)
    sv = Statevector(qc)

    phases_deg = np.degrees([np.angle(amp) for amp in sv.data])
    bar_colors = plt.cm.RdYlGn([c/6 for c in all_cuts])
    ax.bar(range(32), phases_deg, color=bar_colors)
    ax.set_title(gamma_lbl, fontsize=9)
    ax.set_xlabel('Bitstring', fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel('Phase (deg)')
    ax.set_ylim(-200, 200)

plt.suptitle('Effect of γ on Phase Landscape\n(Green=high cut, Red=low cut)', fontsize=12)
plt.tight_layout()
plt.show()

print("γ=0.1:   Phases nearly identical — very little signal for interference")
print("γ=π/6:   Moderate separation — good for QAOA")
print("γ=π/3:   Larger separation")
print("γ=π:     Full range but start wrapping")
print("γ=2π:    Phases wrap all the way around — distinction lost!")

---
### ✏️ Exercise 5.1 — Cost Operator Verification

1. For γ = π/4, compute the **expected phase** (in degrees) for each bitstring using the formula:
   $$\phi(x) = \frac{\gamma}{2}(2 \cdot \text{cut}(x) - |E|)$$
   where $|E|=6$ is the number of edges.

2. Build the circuit and extract the actual phases. Do they match?

3. **Important:** Find all bitstrings with cut=6 and cut=0. What are their phases? By how many degrees do they differ?

In [ ]:
# YOUR CODE HERE
gamma = np.pi / 4
n_edges = len(EDGES)

# 1. Expected phases
print("Expected vs actual phases for γ=π/4:")
print(f"\n{'Bitstring':>10} | {'Cut':>5} | {'Expected phase':>16} | {'Actual phase':>14} | {'Match':>7}")
print("-" * 65)

# Build circuit
qc_ex = QuantumCircuit(N_NODES)
qc_ex.h(range(N_NODES))
qc_ex.compose(cost_operator_circuit(gamma, N_NODES, EDGES), inplace=True)
sv_ex = Statevector(qc_ex)

for bs, amp in zip(all_bitstrings, sv_ex.data):
    cv    = cut_value(bs, EDGES)
    exp_phase = np.degrees(gamma/2 * (2*cv - n_edges))
    act_phase = np.degrees(np.angle(amp))
    match = abs(act_phase - exp_phase) < 0.5
    if cv in [0, 6]:  # show extremes
        print(f"{bs:>10} | {cv:>5} | {exp_phase:>14.2f}° | {act_phase:>12.2f}° | {'✓' if match else '✗'}")

# 3. Phase difference between optimal and worst
cut6_phase = np.degrees(gamma/2 * (2*6 - n_edges))
cut0_phase = np.degrees(gamma/2 * (2*0 - n_edges))
print(f"\nPhase for cut=6: {cut6_phase:.2f}°")
print(f"Phase for cut=0: {cut0_phase:.2f}°")
print(f"Phase difference: {cut6_phase - cut0_phase:.2f}°")
print("This phase difference is what drives constructive/destructive interference.")

---
### ✏️ Exercise 5.2 — Phasor Diagram (Phase as Direction)

Draw a **phasor diagram** showing the phases of all 32 states after $U_C(\pi/6)$:
- Each state is a unit vector (phasor) at angle = its phase
- Color-code by cut value
- States with the same cut value should cluster together

This visualizes WHY high-cut states will interfere constructively after the Mixer.

In [ ]:
# YOUR CODE HERE
gamma = np.pi / 6

qc_phasor = QuantumCircuit(N_NODES)
qc_phasor.h(range(N_NODES))
qc_phasor.compose(cost_operator_circuit(gamma, N_NODES, EDGES), inplace=True)
sv_phasor = Statevector(qc_phasor)

fig, ax = plt.subplots(figsize=(7, 7))

# Draw unit circle
theta_c = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta_c), np.sin(theta_c), 'lightgray', lw=1, zorder=0)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

# Draw each phasor
cmap = plt.cm.RdYlGn
for bs, amp, cv in zip(all_bitstrings, sv_phasor.data, all_cuts):
    phase = np.angle(amp)
    x, y  = np.cos(phase), np.sin(phase)
    color = cmap(cv/6)
    ax.annotate('', xy=(x,y), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2, alpha=0.7))

# Add colorbar legend
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=6))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Cut value')

ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
ax.set_aspect('equal')
ax.set_title('Phasor diagram: all 32 states after H → U_C(γ=π/6)\n(Green=high cut, Red=low cut)')
ax.set_xlabel('Real part'); ax.set_ylabel('Imaginary part')
plt.tight_layout()
plt.show()

print("States cluster by cut value — each cut level has a distinct phase angle.")
print("The Mixer (R_X) will add these phasors, amplifying high-cut clusters.")

---
## ✅ Lab 5 Summary

| Step | Circuit Element | Effect |
|------|----------------|--------|
| 1 | $H^{\otimes 5}$ | Equal superposition of all 32 candidates |
| 2 | $U_C(\gamma)$ | Applies phase $\gamma \cdot \text{cut}(x)$ (up to shift) to each state $|x\rangle$ |
| Phase formula | $\phi(x) = \frac{\gamma}{2}(2\cdot\text{cut}(x) - |E|)$ | Phase proportional to cut value |
| γ effect | Too small → no signal; too large → wraps | Must be tuned by classical optimizer |
| Key property | $U_C$ does NOT change probabilities | Only phases — waiting for Mixer |

**The circuit so far:** 5 H gates + 6×(CNOT-Rz-CNOT) = **5 + 18 = 23 gates** for one QAOA layer.

## 🔭 Preview of Lab 6
Next: **Full QAOA on the 5-Node Max-Cut Problem** — we add the Mixer operator, run multiple rounds, optimize γ and β classically, and measure the final result on the simulator and real hardware.

---
*QOS Lab 5 | Prof. Chansu Yu | Cleveland State University*